# 07 — Atlas: which cell types age, across tissues?

Collects `05a` from every tissue that ran into one table, one row per tissue × cell type, and asks whether the Lung pattern (structural cells keep an age signal, immune cells are batch driven) holds across organs.

Every row uses `cross_age_all_splits`, the same batch-check rule in every tissue. The verdict uses the thresholds fixed in `06_survey` before any new tissue was run:

| Verdict | Rule |
| --- | --- |
| age signal | cross-age ≥ 0.7 and shuffled ≤ 0.6 |
| void | shuffled > 0.6: the pipeline reaches this from noise |
| partial | 0.5 ≤ cross-age < 0.7 |
| batch driven | cross-age < 0.5 |
| not testable | cross-age NaN (missing young or old age groups for this cell type) |

**Tissues share mice** (06: 23 mice in total, median 7 tissues each), so rows are not independent. Pass rates are reported per row, but every comparison is checked by leaving out one tissue at a time.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, mannwhitneyu
from sklearn.metrics import roc_auc_score
from src import aging as ag
from src import style as st    # README figure style

A = ag.paths("atlas")

### 2. Collect every tissue's results

In [ ]:
rows = []
for d in sorted((ag.ROOT / "figure").iterdir()):
    f = d / "age_auroc.csv"
    if d.name == "atlas" or not f.exists():
        continue
    r = pd.read_csv(f, index_col=0)
    rep = pd.read_csv(d / "age_report.csv", index_col=0)
    r = r.join(rep[["auroc_mouse", "n_cells", "n_mice"]])
    r["tissue"] = d.name
    rows.append(r.rename_axis("cell_type").reset_index())
atlas = pd.concat(rows, ignore_index=True)
atlas["compartment"] = atlas["cell_type"].map(ag.compartment)
print(atlas["tissue"].value_counts())

### 3. Verdict per row

In [ ]:
def verdict(r):
    if pd.isna(r["cross_age_all_splits"]):
        return "not testable"
    if r["shuffled"] > 0.6:
        return "void"
    if r["cross_age_all_splits"] >= 0.7:
        return "age signal"
    return "partial" if r["cross_age_all_splits"] >= 0.5 else "batch driven"

atlas["verdict"] = atlas.apply(verdict, axis=1)
cols = ["tissue", "cell_type", "compartment", "logreg", "auroc_mouse", "shuffled",
        "cross_age_all_splits", "verdict", "n_mice", "n_cells"]
atlas = atlas[cols].sort_values(["compartment", "cross_age_all_splits"], ascending=[True, False])
atlas.round(3).to_csv(A.figure("atlas.csv"), index=False)
pd.crosstab(atlas["compartment"], atlas["verdict"], margins=True)

### 3b. Two checks on every row: technical batch, and dominant mice

**Technical batch.** The cross-age check only sees a batch that differs *between age groups*. If all young mice were processed one way and all old mice another, every split still passes (Bladder looked like this: all four cell types 0.89–0.96, leukocytes included). So each row is rerun with **only technical features** per cell (`ag.technical_adata`: log total counts, log genes per cell, ribosomal, dissociation and hemoglobin fractions). If those alone pass the cross-age check (`tech_cross_age` ≥ 0.7), the row is `tech_confounded`: its age signal can't be told apart from processing. Old cells can differ technically for real reasons, so this is a warning, not proof.

**Dominant mice.** A row whose cell type gets over half its cells from one mouse (`outlier_mice.csv` from 04) is rerun without that mouse. The main table keeps every mouse (the 300-cell cap already limits each); `flips` marks rows whose verdict changes without it.

In [ ]:
extra = []
for tissue in sorted(atlas["tissue"].unique()):
    a = sc.read_h5ad(ag.paths(tissue).processed)
    tech = ag.technical_adata(a)
    flags = pd.read_csv(ag.paths(tissue).figure("outlier_mice.csv"), index_col=0)
    for ct in atlas.loc[atlas["tissue"] == tissue, "cell_type"]:
        kw = ag.main_kw(ct, tissue)
        row = {"tissue": tissue, "cell_type": ct,
               "tech_auroc": ag.age_auc(tech, ct, **kw),
               "tech_cross_age": ag.cross_age_auc(tech, ct, **kw)}
        if ct in flags.index and flags.loc[ct, "top_mouse"] not in kw.get("drop_mice", []):
            m = flags.loc[ct, "top_mouse"]
            kw2 = {"drop_mice": kw.get("drop_mice", []) + [m]}
            row.update(dominant_mouse=m, dominant_share=flags.loc[ct, "share"],
                       logreg_without=ag.age_auc(a, ct, **kw2),
                       cross_age_without=ag.cross_age_auc(a, ct, **kw2))
        extra.append(row)
    del a, tech
atlas = atlas.merge(pd.DataFrame(extra), on=["tissue", "cell_type"], how="left")
atlas["tech_confounded"] = atlas["tech_cross_age"] >= 0.7
has_dom = atlas["dominant_mouse"].notna()
atlas["verdict_without"] = np.where(has_dom, atlas.assign(cross_age_all_splits=atlas["cross_age_without"])
                                    .apply(verdict, axis=1), None)
atlas["flips"] = has_dom & (atlas["verdict_without"] != atlas["verdict"])
atlas.round(3).to_csv(A.figure("atlas.csv"), index=False)


### 3c. Technical batch per tissue

Median of `tech_cross_age` over a tissue's rows, and how many rows are confounded.

In [ ]:
atlas.groupby("tissue").agg(rows=("cell_type", "size"),
                            median_cross_age=("cross_age_all_splits", "median"),
                            median_tech_cross_age=("tech_cross_age", "median"),
                            tech_confounded=("tech_confounded", "sum"),
                            age_signal=("verdict", lambda v: (v == "age signal").sum())).round(2)

### 3d. Rows dominated by one mouse, with and without it

In [ ]:
atlas.loc[has_dom, ["tissue", "cell_type", "dominant_mouse", "dominant_share", "logreg", "logreg_without",
                    "cross_age_all_splits", "cross_age_without", "verdict", "verdict_without", "flips"]].round(2)

### 4. Check the ambiguous labels against marker genes

Some TMS labels don't say what the cell is. Before they count toward a compartment, their mean expression of compartment markers is compared with clear cell types of the same tissue: stromal *Col1a1*, *Dcn*, *Pdgfra*; endothelial *Pecam1*, *Cdh5*; smooth muscle *Acta2*; immune *Ptprc*; epithelial *Epcam*, *Krt8*.

In [ ]:
markers = ["Col1a1", "Dcn", "Pdgfra", "Pecam1", "Cdh5", "Acta2", "Ptprc", "Epcam", "Krt8"]
check = {"Bladder": ["bladder cell", "bladder urothelial cell", "endothelial cell", "leukocyte"],
         "Kidney": ["kidney cortex artery cell", "brush cell", "fenestrated cell", "kidney mesangial cell", "macrophage"]}
tables = []
for tissue, cts in check.items():
    if not ag.paths(tissue).processed.exists():
        continue
    a = sc.read_h5ad(ag.paths(tissue).processed)
    a = a[a.obs["cell_ontology_class"].isin(cts), [g for g in markers if g in a.var_names]]
    m = sc.get.aggregate(a, by="cell_ontology_class", func="mean")
    tables.append(pd.DataFrame(m.layers["mean"], index=tissue + ": " + m.obs["cell_ontology_class"].astype(str),
                               columns=m.var_names))
marker_table = pd.concat(tables).round(2)
marker_table

### 5. Heatmap: cross-age AUROC, cell type × tissue

Rows grouped by compartment. A cell type name shared by several tissues (e.g. "endothelial cell", "macrophage") is one row. Grey = not run in that tissue. A dot marks a voided row (shuffled > 0.6).

In [ ]:
st.atlas_heatmap(atlas, A.figure("atlas_cross_age.png"))

### 6. Pass rate per compartment

Voided and not-testable rows are left out. `rate` is the fraction of tissue × cell type rows with an age signal; `median_cross_age` uses every testable row.

In [ ]:
def rates(d):
    r = d.groupby("compartment").agg(rows=("verdict", "size"), tissues=("tissue", "nunique"),
                                     age_signal=("verdict", lambda v: (v == "age signal").sum()),
                                     median_cross_age=("cross_age_all_splits", "median"))
    r["rate"] = (r["age_signal"] / r["rows"]).round(2)
    return r.round(2).sort_values("rate", ascending=False)

use = atlas[~atlas["verdict"].isin(["void", "not testable"])]
clean = use[~use["tech_confounded"]]
print("all testable rows"); display(rates(use))
print("without technically confounded rows"); display(rates(clean))

### 7. Structural vs. immune: does the Lung pattern hold?

"Structural" = stromal + endothelial + epithelial. Two tests on the rows (Fisher on pass/fail, Mann-Whitney on cross-age AUROC), then the same comparison with each tissue left out in turn. Rows share mice, so the p-values are optimistic; the leave-one-tissue-out column says whether a single organ carries the result.

In [ ]:
def compare(d):
    s = d[d["compartment"].isin(["stromal", "endothelial", "epithelial"])]
    im = d[d["compartment"] == "immune"]
    table = [[(s["verdict"] == "age signal").sum(), (s["verdict"] != "age signal").sum()],
             [(im["verdict"] == "age signal").sum(), (im["verdict"] != "age signal").sum()]]
    return {"structural_rate": round(table[0][0] / max(len(s), 1), 2),
            "immune_rate": round(table[1][0] / max(len(im), 1), 2),
            "fisher_p": fisher_exact(table)[1] if len(s) and len(im) else np.nan,
            "mwu_p": mannwhitneyu(s["cross_age_all_splits"], im["cross_age_all_splits"])[1] if len(s) and len(im) else np.nan,
            "n_structural": len(s), "n_immune": len(im)}

loo = {"all tissues": compare(use)}
for t in sorted(use["tissue"].unique()):
    loo[f"without {t}"] = compare(use[use["tissue"] != t])
loo["all, no tech-confounded rows"] = compare(clean)
for t in sorted(clean["tissue"].unique()):
    loo[f"no tech-confounded, without {t}"] = compare(clean[clean["tissue"] != t])
loo = pd.DataFrame(loo).T
loo.to_csv(A.figure("atlas_structural_vs_immune.csv"))
loo

### 8. The same cell type in several tissues

A cell type name present in 3+ tissues: does it pass everywhere, nowhere, or depend on the organ? These rows share mice across tissues, so agreement can come from a mouse-level batch effect as well as from shared biology.

In [ ]:
multi = atlas.groupby("cell_type")["tissue"].nunique()
shared = atlas[atlas["cell_type"].isin(multi[multi >= 3].index)]
shared.pivot_table(index="cell_type", columns="tissue", values="cross_age_all_splits").round(2)

### 9. Rows with an age signal

The rows to carry into gene analysis (08–09).

In [ ]:
passing = atlas[atlas["verdict"] == "age signal"].sort_values("cross_age_all_splits", ascending=False)
passing.round(2)

### 10. Stricter tests of the structural-versus-immune result

Section 7 compared rows with Fisher and Mann–Whitney tests, which treat every row as an independent sample. They are not: rows of one tissue share the same mice, and some tissues (Marrow, Spleen) contribute only immune rows. A tissue whose rows all score low for a tissue-wide reason (few or odd mice, a strong batch) then lowers the immune average without saying anything about immune cells. Three stricter versions:

- **Within-tissue permutation.** The structural/immune label is shuffled only among rows of the same tissue, 20,000 times. Tissues with a single compartment then contribute nothing, and only differences *inside* a tissue count.
- **Tissue fixed effects.** A linear model of cross-age AUROC on structural (0/1), log cell count and mouse count, with one intercept per tissue. Also asks whether row size (power) explains the gap.
- **Direction per tissue.** In how many tissues that have both kinds of rows does the structural mean exceed the immune mean?

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import spearmanr, binomtest
STRUCT = ["stromal", "endothelial", "epithelial"]

def within_tissue_test(d, n_perm=20000, seed=0):
    d = d[d["compartment"].isin(STRUCT + ["immune"])]
    s = d["compartment"].isin(STRUCT).to_numpy()
    passed = d["verdict"].eq("age signal").to_numpy()
    xa = d["cross_age_all_splits"].to_numpy()

    def stat(lab):
        return passed[lab].mean() - passed[~lab].mean(), xa[lab].mean() - xa[~lab].mean()

    obs = stat(s)
    rng = np.random.default_rng(seed)
    groups = [np.where(d["tissue"].to_numpy() == t)[0] for t in d["tissue"].unique()]
    null = np.empty((n_perm, 2))
    for b in range(n_perm):
        lab = s.copy()
        for g in groups:
            lab[g] = rng.permutation(s[g])
        null[b] = stat(lab)
    p = [(1 + (null[:, k] >= obs[k]).sum()) / (n_perm + 1) for k in range(2)]
    fe = smf.ols("xa ~ structural + np.log10(n_cells) + n_mice + C(tissue)",
                 d.assign(xa=xa, structural=s.astype(int))).fit()
    return {"rows": len(d), "pass_rate_diff": obs[0], "p_within_tissue_pass": p[0],
            "mean_cross_age_diff": obs[1], "p_within_tissue_cross_age": p[1],
            "fixed_effect_structural": fe.params["structural"], "fixed_effect_p": fe.pvalues["structural"],
            "fixed_effect_log_cells_p": fe.pvalues["np.log10(n_cells)"], "fixed_effect_n_mice_p": fe.pvalues["n_mice"]}

strict = pd.DataFrame({"all testable rows": within_tissue_test(use),
                       "no tech-confounded rows": within_tissue_test(clean)}).T
strict.round(4).to_csv(A.figure("atlas_strict_tests.csv"))
display(strict.round(4))

d = clean[clean["compartment"].isin(STRUCT + ["immune"])]
per_tissue = d.assign(structural=d["compartment"].isin(STRUCT)).pivot_table(
    index="tissue", columns="structural", values="cross_age_all_splits", aggfunc="mean").dropna()
per_tissue.columns = ["immune", "structural"]
k = int((per_tissue["structural"] > per_tissue["immune"]).sum())
print(f"structural mean higher in {k} of {len(per_tissue)} tissues with both "
      f"(one-sided sign test p = {binomtest(k, len(per_tissue), alternative='greater').pvalue:.3f})")
for col in ["n_cells", "n_mice"]:
    r = spearmanr(use[col], use["cross_age_all_splits"])
    print(f"cross-age vs {col}: Spearman rho = {r[0]:.2f}, p = {r[1]:.3f}")
per_tissue.round(2)

### 11. Three more checks per row: sex, and the uncertainty of each score

**Sex is tied to age group in every tissue** (06: the 21m mice are all female, the 24m and 30m mice all male, most 1m mice male). Since the critical review, the main model uses every highly variable gene **except** the six sex-chromosome genes (*Xist*, *Tsix*, *Ddx3y*, *Eif2s3y*, *Kdm5d*, *Uty*; `ag.SEX_PATTERN`), which `ag.prep` now drops by default. Before, they were in, and a model could separate a female age group from a male one by sex. Three checks:

- `sex_only_cross_age`: the cross-age check with the donor's sex as the **only** feature (`ag.sex_adata`). If sex alone reaches ≥ 0.7, that row's verdict cannot be told apart from sex.
- `with_sex_genes_cross_age`: the main check with the six sex-chromosome genes put back (`ag.feature_mask(a, "hvg")`), i.e. the pre-review default. The main check removes them, but autosomal genes that differ by sex remain, so removal is a partial fix.
- `males_only_cross_age`: the main check on male mice only, NaN where males do not give two young and two old ages. The strongest test, where possible.

**How sure is each cross-age score?** With 2–6 test mice per split, an AUROC can move a lot depending on which mice happened to be tested. `ci90_low` / `ci90_high` resample whole test mice (`ag.bootstrap_cross_age`, 1,000 draws). Training mice are held fixed, so these intervals are narrower than the full uncertainty, and a class with one test mouse adds no spread.

In [ ]:
strict_rows = []
for tissue in sorted(atlas["tissue"].unique()):
    a = sc.read_h5ad(ag.paths(tissue).processed)
    sx = ag.sex_adata(a)
    with_sex = ag.feature_mask(a, "hvg")   # pre-review default: sex-chromosome genes in
    males = a.obs.loc[a.obs["sex"].astype(str) == "male", "mouse.id"].astype(str).unique().tolist()
    for ct in atlas.loc[atlas["tissue"] == tissue, "cell_type"]:
        kw = ag.main_kw(ct, tissue)
        parts = ag.cross_age_scores(a, ct, **kw)
        lo, hi = ag.bootstrap_cross_age(parts)
        strict_rows.append({
            "tissue": tissue, "cell_type": ct,
            "cross_age_check": np.mean([roc_auc_score(q["y"], q["score"]) for q in parts]) if parts else np.nan,
            "ci90_low": lo, "ci90_high": hi,
            "sex_only_cross_age": ag.cross_age_auc(sx, ct, **kw),
            "with_sex_genes_cross_age": ag.cross_age_auc(a, ct, features=with_sex, **kw),
            "males_only_cross_age": ag.cross_age_auc(a, ct, keep_mice=males, **kw)})
    del a, sx
strict_rows = pd.DataFrame(strict_rows)
atlas = atlas.drop(columns=[c for c in strict_rows.columns if c in atlas.columns and c not in ["tissue", "cell_type"]])
atlas = atlas.merge(strict_rows, on=["tissue", "cell_type"], how="left")
# the refactored cross_age_scores must reproduce the stored score exactly
assert np.allclose(atlas["cross_age_check"], atlas["cross_age_all_splits"], atol=1e-3, equal_nan=True)
atlas = atlas.drop(columns="cross_age_check")
atlas.round(3).to_csv(A.figure("atlas.csv"), index=False)

In [ ]:
sig = atlas[atlas["verdict"] == "age signal"]
testable = atlas[~atlas["verdict"].isin(["not testable"])]
print(pd.Series({
    "age-signal rows": len(sig),
    "  90% interval entirely above 0.5": int((sig["ci90_low"] > 0.5).sum()),
    "  90% interval entirely above 0.7": int((sig["ci90_low"] >= 0.7).sum()),
    "  sex alone also passes (>= 0.7)": int((sig["sex_only_cross_age"] >= 0.7).sum()),
    "  still >= 0.7 with sex-chromosome genes back in": int((sig["with_sex_genes_cross_age"] >= 0.7).sum()),
    "  testable in males only": int(sig["males_only_cross_age"].notna().sum()),
    "  still >= 0.7 in males only": int((sig["males_only_cross_age"] >= 0.7).sum()),
    "all testable rows where sex alone passes": int((testable["sex_only_cross_age"] >= 0.7).sum()),
    "median |change| with sex genes, all rows": round(float((testable["with_sex_genes_cross_age"] - testable["cross_age_all_splits"]).abs().median()), 3),
}).to_string())
sig[["tissue", "cell_type", "compartment", "cross_age_all_splits", "ci90_low", "ci90_high",
     "sex_only_cross_age", "with_sex_genes_cross_age", "males_only_cross_age", "tech_confounded"]].round(2)

### 12. Sex genes hide signal as well as fake it

The critical review found that removing the six sex-chromosome genes changed 12 verdicts, all **upward** (partial → ≥ 0.7), mostly Marrow immune and progenitor rows. Sex is tied to age group, so in some splits the training ages pair, say, male young with female old while the test ages pair them the other way; a model that leaned on *Xist* is then inverted on the test set. Removal is therefore now the default, and this section shows the reverse view: which verdicts would differ with the sex genes put back (`with_sex_genes_cross_age`), and the within-tissue comparison of section 10 on both columns.

In [ ]:
import numpy as np, pandas as pd
d0 = pd.read_csv(A.figure("atlas.csv"))
d0 = d0[~d0["verdict"].isin(["void", "not testable"])]
flip = d0[(d0["with_sex_genes_cross_age"] >= 0.7) != (d0["cross_age_all_splits"] >= 0.7)]
display(flip[["tissue", "cell_type", "compartment", "cross_age_all_splits", "with_sex_genes_cross_age",
              "males_only_cross_age"]].round(2))
STRUCT = ["stromal", "endothelial", "epithelial"]

def compare_within(d, col, n_perm=20000, seed=0):
    d = d[d["compartment"].isin(STRUCT + ["immune"])]
    s = d["compartment"].isin(STRUCT).to_numpy()
    ps, xa = (d[col] >= 0.7).to_numpy(), d[col].to_numpy()
    stat = lambda lab: (ps[lab].mean() - ps[~lab].mean(), xa[lab].mean() - xa[~lab].mean())
    obs = stat(s)
    rng = np.random.default_rng(seed)
    groups = [np.where(d["tissue"].to_numpy() == t)[0] for t in d["tissue"].unique()]
    null = np.empty((n_perm, 2))
    for b in range(n_perm):
        lab = s.copy()
        for g in groups:
            lab[g] = rng.permutation(s[g])
        null[b] = stat(lab)
    p = [(1 + (null[:, k] >= obs[k]).sum()) / (n_perm + 1) for k in range(2)]
    return {"rows": len(d), "structural_rate": ps[s].mean(), "immune_rate": ps[~s].mean(),
            "pass_rate_diff": obs[0], "p_within_tissue_pass": p[0],
            "mean_cross_age_diff": obs[1], "p_within_tissue_cross_age": p[1]}

clean0 = d0[~d0["tech_confounded"].astype(bool)]
out = pd.DataFrame({f"{name}, {col}": compare_within(d, col)
                    for name, d in [("all testable", d0), ("no tech-confounded", clean0)]
                    for col in ["cross_age_all_splits", "with_sex_genes_cross_age"]}).T
out.round(4).to_csv(A.figure("atlas_strict_tests_sex.csv"))
out.round(3)

In [ ]:
st.sex_gene_slope(atlas, A.figure("atlas_sex_genes.png"))   # README figure

### Findings

70 tissue × cell type rows from 9 tissues (23 mice in total; tissues share mice). Thresholds as fixed in 06. Since the critical review, every model leaves out the six sex-chromosome genes (sections 11–12); the pre-review verdicts kept them in. Sections 10–12 were added in that review.

- **Verdicts:** 40 age signal, 19 partial, 4 batch driven, 7 void (with sex genes: 29, 28, 6, 7). 11 partial rows became age signals (Marrow 7, Spleen B cell, Liver Kupffer cell, Heart leukocyte, Lung bronchial smooth muscle) and two batch-driven Marrow rows became partial. Two Kidney rows swapped through the shuffled control, which sits at the 0.6 cutoff in both: podocyte void → age signal (0.61 → 0.59), macrophage age signal → void (0.60 → 0.62). Read those two as threshold cases.
- **Structural rows score higher than immune rows of the same tissue, but they do not pass more often.** Over rows, structural pass 0.74 vs immune 0.56 (Fisher p = 0.25; with sex genes 0.70 vs 0.30). Within tissue (section 10) the pass-rate difference is not significant (p = 0.28; 0.20 without technically confounded rows), while the mean cross-age AUROC is higher in structural rows by 0.10 (permutation p = 0.013; 0.11, p = 0.002 without technically confounded rows; tissue fixed effect +0.12, p = 0.008, not explained by cell or mouse count). Structural is higher in 4 of 6 tissues with both (sign test p = 0.34). Lung carries the gap (0.87 vs 0.58): without Lung the immune pass rate (0.78) exceeds the structural one (0.70).
- **Sex genes hid immune signal (section 12).** With the six sex-chromosome genes put back, 11 of the 40 age signals fall below 0.7 and none rises; the immune pass rate drops from 0.56 to 0.30. Sex is tied to age group, so a split that pairs male young with female old in training and the reverse in test inverts a model leaning on *Xist*.
- **Sex does not fake the passes.** Sex alone never reaches 0.7 in any row (max 0.68, Lung fibroblast). Of the 22 age-signal rows testable in male mice only, 19 stay ≥ 0.7 (Kidney collecting duct 0.59, brush cell 0.68, Spleen T 0.69 drop). Autosomal genes that differ by sex remain, so removal is a partial fix.
- **The passes are not a lucky draw of test mice.** All 40 age-signal rows have a 90% interval (test mice resampled) above 0.5; 35 are entirely above 0.7, and the 5 that reach below sit near the cutoff (Kidney brush cell, loop of Henle ascending limb; Liver Kupffer cell; Marrow monocyte, promonocyte). Training mice are held fixed, so these intervals are too narrow, not too wide.
- **Immune cells pass in most tissues.** Limb_Muscle B (0.83), T (0.86) and macrophage (0.76); Spleen T (0.82), NK T (0.85), B (0.77); Marrow macrophage (0.77), naive T (0.75), plasma cell (0.72), monocyte (0.71); Bladder leukocyte (0.90); Liver NK (0.79); Lung alveolar macrophage (0.79). Lung lymphocytes and monocytes stay at 0.45–0.60, so the Lung pattern that started the project is a Lung property, not an immune one.
- **Technical batch is real in Bladder.** Technical features alone (depth, genes per cell, ribosomal, dissociation, blood fractions) pass the cross-age check in 2 of 4 Bladder rows, so Bladder's uniform 0.90–0.96 is partly processing. Heart fibroblasts (0.74) and Marrow precursor B cells (0.83) are the other flagged age-signal rows. 36 of 40 age-signal rows survive this check.
- **Dominant mice change two rows.** Without 1-M-63, Liver sinusoidal endothelium goes from partial (0.60) to an age signal (0.84). Without 30-M-3, Liver Kupffer cells, one of the new age signals, fall back to partial (0.71 → 0.67), so that pass rests on one mouse. Heart coronary-artery endothelium no longer flips (0.64 → 0.61 without 30-M-2, partial both ways). 30-M-2, excluded from Lung myeloid cells, also dominates cell types in Heart and Marrow, which fits a body-wide condition in that mouse.
- **Labels check out against markers:** "bladder cell" is mesenchymal (*Col1a1*, *Dcn*), "kidney cortex artery cell" endothelial (*Pecam1*, *Cdh5*), "brush cell" epithelial (*Krt8*, *Epcam*), mesangial cells smooth-muscle-like (*Acta2*).
- **Not testable:** Fat, Pancreas, Skin, Large_Intestine (no young mice), Thymus (one young mouse), Trachea (no old mice), Mammary_Gland (young are all 3m).